# Free GPU Imager — SDXL Base 1.0 on Colab T4 + FastAPI + ngrok

**ACHIEVEMOR / FOAI — `~/foai/free-gpu-imager/`**

Run a state-of-the-art image generation model on Google Colab's free T4 GPU (15 GB VRAM, no credit card), expose it as a REST API via FastAPI + ngrok, and call it from any client for $0.00.

**Model:** `stabilityai/stable-diffusion-xl-base-1.0` (3.5B params, 1024×1024 native).

## Before you run anything

1. **Runtime → Change runtime type → T4 GPU** (the free tier offers this; do not skip).
2. **Tools → Secrets** — add two secrets, both with notebook access toggled on:
   - `HF_TOKEN` — get one at https://huggingface.co/settings/tokens (read access is enough)
   - `NGROK_AUTH_TOKEN` — get one at https://dashboard.ngrok.com/get-started/your-authtoken (free tier is fine)
3. **Runtime → Run all**.

Once it finishes, you'll see a public URL printed at the bottom — that's your live REST API. The Swagger UI is at `<URL>/docs`.

## 1 · Pull tokens from Colab Secrets (no hard-coding)

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

assert HF_TOKEN, 'Add HF_TOKEN to Tools → Secrets and toggle notebook access ON.'
assert NGROK_AUTH_TOKEN, 'Add NGROK_AUTH_TOKEN to Tools → Secrets and toggle notebook access ON.'
print('Tokens loaded from Colab Secrets — never written to the notebook.')

## 2 · Install pinned dependencies

Versions verified against PyPI 2026-04-27. Colab provides torch + CUDA — we don't pin torch.

In [ ]:
%pip install -q --upgrade \
    diffusers==0.37.1 \
    'transformers==4.57.6' \
    accelerate==1.13.0 \
    safetensors \
    'huggingface_hub<2.0' \
    fastapi==0.136.1 \
    'uvicorn[standard]' \
    pyngrok==8.1.1 \
    gradio==6.13.0 \
    nest_asyncio \
    pillow

# Note: transformers is pinned to 4.x because diffusers 0.37.1 (released
# 2026-03-25) was authored against the transformers 4.x API. transformers 5.x
# is a major version bump with breaking changes that would break diffusers
# imports at runtime even though pip will happily install it.

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in to Hugging Face — fast downloads enabled.')

## 3 · Load the SDXL pipeline (~7 GB download, ~2-4 min on free tier)

In [ ]:
import torch
from diffusers import DiffusionPipeline

MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'

pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
)
pipe = pipe.to('cuda')

# T4 only has 15 GB VRAM — these reduce peak usage with negligible quality cost.
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

print(f'Pipeline ready on {pipe.device}. dtype = {pipe.unet.dtype}.')

### 3.1 · Smoke test — generate one image

In [ ]:
from IPython.display import display

TEST_PROMPT = (
    'a tactical hawk character mascot with an orange LED visor, gold ANG chest patch, '
    'matte black hoodie, cinematic studio lighting, 4K, photorealistic'
)

test_image = pipe(
    prompt=TEST_PROMPT,
    num_inference_steps=30,
    guidance_scale=7.5,
).images[0]

test_image.save('/content/test.png')
display(test_image)
print('Saved to /content/test.png')

## 3.2 · Wrap it as a REST API (FastAPI + ngrok)

Defines `GET /health` and `POST /generate`. The ngrok tunnel gives you a public URL callable from any React, mobile, or Python client.

In [ ]:
import base64
import io
import threading
import time
from typing import Optional

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(
    title='Free GPU Imager',
    description='SDXL Base 1.0 on Colab T4 — FOAI free image gen',
    version='1.0.0',
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['GET', 'POST'],
    allow_headers=['*'],
)


class GenerateRequest(BaseModel):
    prompt: str = Field(..., min_length=1, max_length=2000)
    negative_prompt: Optional[str] = Field(default=None, max_length=2000)
    num_inference_steps: int = Field(default=30, ge=10, le=60)
    guidance_scale: float = Field(default=7.5, ge=1.0, le=15.0)
    width: int = Field(default=1024, ge=512, le=1024)
    height: int = Field(default=1024, ge=512, le=1024)
    seed: Optional[int] = Field(default=None, ge=-1)


class GenerateResponse(BaseModel):
    image_base64: str
    seed: int
    elapsed_seconds: float
    model: str


@app.get('/health')
def health():
    return {
        'ok': True,
        'model': MODEL_ID,
        'device': str(pipe.device),
        'dtype': str(pipe.unet.dtype),
    }


@app.post('/generate', response_model=GenerateResponse)
def generate(req: GenerateRequest):
    if req.width % 8 or req.height % 8:
        raise HTTPException(status_code=400, detail='width/height must be divisible by 8')

    seed = req.seed if (req.seed is not None and req.seed >= 0) else int(torch.seed() % (2 ** 31))
    generator = torch.Generator(device='cuda').manual_seed(seed)

    t0 = time.time()
    image = pipe(
        prompt=req.prompt,
        negative_prompt=req.negative_prompt,
        num_inference_steps=req.num_inference_steps,
        guidance_scale=req.guidance_scale,
        width=req.width,
        height=req.height,
        generator=generator,
    ).images[0]
    elapsed = time.time() - t0

    buf = io.BytesIO()
    image.save(buf, format='PNG')
    return GenerateResponse(
        image_base64=base64.b64encode(buf.getvalue()).decode('ascii'),
        seed=seed,
        elapsed_seconds=round(elapsed, 2),
        model=MODEL_ID,
    )


# Open the tunnel before starting the server so the URL prints first.
# NGROK_DOMAIN is the owner's reserved free-tier domain — same URL across restarts,
# so client scripts can pin to it without rediscovery.
NGROK_DOMAIN = 'avoid-capably-broadcast.ngrok-free.dev'
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
tunnel = ngrok.connect(8000, 'http', domain=NGROK_DOMAIN)
PUBLIC_URL = tunnel.public_url

print('=' * 60)
print(f'  Public API : {PUBLIC_URL}')
print(f'  Swagger UI : {PUBLIC_URL}/docs')
print(f'  Health     : {PUBLIC_URL}/health')
print('=' * 60)


def _serve():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')


_server_thread = threading.Thread(target=_serve, daemon=True)
_server_thread.start()
time.sleep(2)
print('FastAPI server thread started. Hit the Swagger UI link above to try it.')

## 4 · Optional Gradio UI in this notebook

If you want a clickable browser UI alongside the API, run this cell. Gradio's own `share=True` gives you a second public URL for the UI; the FastAPI tunnel above is the one your apps should hit.

In [ ]:
import gradio as gr


def gradio_generate(prompt, negative_prompt, steps, scale, seed, width, height):
    seed_int = None if seed == -1 else int(seed)
    used_seed = seed_int if seed_int is not None else int(torch.seed() % (2 ** 31))
    generator = torch.Generator(device='cuda').manual_seed(used_seed)
    image = pipe(
        prompt=prompt,
        negative_prompt=(negative_prompt or None),
        num_inference_steps=int(steps),
        guidance_scale=float(scale),
        width=int(width),
        height=int(height),
        generator=generator,
    ).images[0]
    return image, used_seed


ui = gr.Interface(
    fn=gradio_generate,
    inputs=[
        gr.Textbox(label='Prompt', lines=3, value=TEST_PROMPT),
        gr.Textbox(label='Negative prompt', lines=2, value=''),
        gr.Slider(10, 60, value=30, step=1, label='Inference steps'),
        gr.Slider(1.0, 15.0, value=7.5, step=0.1, label='Guidance scale'),
        gr.Number(value=-1, label='Seed (-1 = random, copy from output to reproduce)'),
        gr.Slider(512, 1024, value=1024, step=64, label='Width'),
        gr.Slider(512, 1024, value=1024, step=64, label='Height'),
    ],
    outputs=[
        gr.Image(label='Output', type='pil'),
        gr.Number(label='Seed used'),
    ],
    title='Free GPU Imager — SDXL Base 1.0',
    description='Free image generation on Colab T4. Paste a seed to reproduce; -1 randomizes.',
    allow_flagging='never',
)
ui.launch(share=True, inline=False)

## 5 · Calling the API from anywhere

The reserved domain is fixed: **`https://avoid-capably-broadcast.ngrok-free.dev`** — same across Colab restarts.

**Python / requests:**
```python
import base64, requests
from pathlib import Path

API = 'https://avoid-capably-broadcast.ngrok-free.dev'
r = requests.post(f'{API}/generate', json={
    'prompt': 'a hawk in tactical gear, photoreal',
    'num_inference_steps': 30,
    'guidance_scale': 7.5,
}, timeout=120).json()
Path('out.png').write_bytes(base64.b64decode(r['image_base64']))
```

**React / fetch:**
```ts
const res = await fetch(`${API}/generate`, {
  method: 'POST',
  headers: { 'Content-Type': 'application/json' },
  body: JSON.stringify({ prompt, num_inference_steps: 30 }),
});
const { image_base64 } = await res.json();
// <img src={`data:image/png;base64,${image_base64}`} />
```

**Curl:**
```bash
API=https://avoid-capably-broadcast.ngrok-free.dev
curl -X POST "$API/generate" -H 'Content-Type: application/json' \
  -d '{"prompt":"a hawk in tactical gear, photoreal"}' \
  | jq -r .image_base64 | base64 -d > out.png
```

## 6 · Notes & limits

- **Reserved domain is pinned.** The tunnel always opens on `avoid-capably-broadcast.ngrok-free.dev`. If you start another Colab session while this one is still alive, the second `ngrok.connect()` will fail because the domain is in use — disconnect the first tunnel or close that Colab.
- **Free Colab idles after ~90 min** of inactivity. The runtime stops; the reserved domain stays yours but won't answer until you restart and re-run.
- **One request at a time.** SDXL on a T4 saturates the GPU; concurrent requests will queue. Wrap a `threading.Lock` around `pipe(...)` for explicit serialization if you need it.
- **First call is slow** (~25 s). Subsequent calls run in ~10-15 s for 30 steps at 1024×1024.
- **VRAM headroom.** Loaded pipeline uses ~7 GB; inference peaks around 11 GB. Stay below 1024×1024 if you hit OOM.
- **Swap models** by changing `MODEL_ID` and rerunning the load cell. Tested alternatives that fit on T4: `segmind/SSD-1B`, `stabilityai/sdxl-turbo`, `stabilityai/stable-diffusion-3-medium-diffusers` (gated).

Built for `~/foai/free-gpu-imager/` — see `README.md` for FOAI integration patterns and `examples/sqwaadrun_portraits.py` for batch portrait regeneration.